# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lokeshtiwari723/Proto-ex/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents one page-level search/content record for the analysis window.

The ranking task is to prioritize pages that should be reviewed first for possible search-performance improvement.

The feature window is February 2026, and March 2026 is used as the following label window.

The target is a search-performance outcome used as a proxy for ranking priority. The exact label field and aggregation will be verified from the warehouse data below.

I will exclude any field that is only known after the decision cutoff or is derived from the future label window, because it would create leakage.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-04 warehouse setup

%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Keep the token out of SQL text.
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("Warehouse connection setup complete.")
print("Feature window: February 2026")
print("Label window: March 2026")

Warehouse connection setup complete.
Feature window: February 2026
Label window: March 2026


In [15]:
# ML-04: inspect the actual warehouse schema
print("FACT columns:")
display(con.sql(f"DESCRIBE SELECT * FROM {FEB}").df())

print("\nCONTENT dimension columns:")
display(con.sql(f"DESCRIBE SELECT * FROM {DIM}").df())

print("\nCLIENT dimension columns:")
display(con.sql(f"DESCRIBE SELECT * FROM {CLI}").df())

FACT columns:


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



CONTENT dimension columns:


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None



CLIENT dimension columns:


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [16]:
# ML-04: print all warehouse column names clearly

fact_cols = con.sql(f"DESCRIBE SELECT * FROM {FEB}").df()["column_name"].tolist()

print("All fact table columns:")
for i, col in enumerate(fact_cols, 1):
    print(f"{i}. {col}")

All fact table columns:
1. report_date
2. client_hash_id
3. content_hash_id
4. client_has_gsc
5. client_has_ga4
6. gsc_data_available
7. ga4_data_available
8. gsc_impressions
9. gsc_clicks
10. gsc_sum_position
11. gsc_avg_position
12. ga4_pageviews
13. ga4_sessions
14. ga4_users
15. ga4_engaged_sessions
16. ga4_total_engagement_sec
17. sessions_organic
18. sessions_direct
19. sessions_referral
20. sessions_social
21. sessions_paid
22. sessions_ai
23. ai_chatgpt
24. ai_perplexity
25. ai_gemini
26. ai_copilot
27. ai_claude
28. ai_meta
29. ai_other
30. scroll_events
31. month


In [17]:
# ML-04: inspect content metadata columns

content_cols = con.sql(
    f"DESCRIBE SELECT * FROM {DIM}"
).df()["column_name"].tolist()

print("All content dimension columns:")
for i, col in enumerate(content_cols, 1):
    print(f"{i}. {col}")

All content dimension columns:
1. client_hash_id
2. content_hash_id
3. keyword_hash_id
4. url_hash_id
5. keyword_char_count
6. keyword_token_count
7. url_char_count
8. content_created_date
9. content_updated_date
10. content_type
11. search_volume
12. competition
13. competition_level
14. cpc
15. main_intent
16. backlinks
17. category_count
18. keyword_created_date
19. provider_used
20. model_used
21. char_count
22. word_count
23. last_optimized_date
24. optimization_eligible_date
25. is_published
26. is_deleted


## 2. Fields: feature / label / context / excluded

### Features — February 2026

1. `gsc_impressions` — February search impressions; knowable at the decision moment from the completed February feature window.
2. `gsc_clicks` — February search clicks; knowable at the decision moment from the completed February feature window.
3. `gsc_avg_position` — February average search position; knowable at the decision moment from the completed February feature window.
4. `word_count` — content length from the content dimension; knowable before the decision cutoff.
5. `content_created_date` — content age can be derived from the creation date; knowable before the decision cutoff.

### Label / proxy

The label is derived from the March 2026 outcome window. March search clicks and impressions are used to calculate the observed CTR outcome, with search position used to form the position-based comparison. March data is label information and is not used as a February feature.

### Context

- `client_hash_id` — identifies the client grouping used for analysis and validation.
- `content_hash_id` — identifies the content record.
- `report_date` / `month` — identifies the observation window.
- `gsc_data_available` — indicates whether GSC data is available for the observation.
- `client_has_gsc` — indicates whether the client has GSC connected.
- `client_has_ga4` / `ga4_data_available` — context about GA4 availability.

### Excluded

- March `gsc_clicks`, `gsc_impressions`, and other March performance fields — excluded because they belong to the future label window and would cause leakage.
- `trend_direction` or any other label-derived field — excluded because it would directly reveal the outcome.
- `content_updated_date` / `last_optimized_date` when they reflect events after the decision cutoff — excluded if the value is not historically available at the cutoff.
- `client_hash_id`, `content_hash_id`, and other identifiers — excluded from predictive features because they identify records rather than provide a meaningful pre-cutoff ranking signal.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

### Five-feature frame — February 2026

1. `gsc_impressions` — February search impressions; knowable at the decision moment because the complete February feature window has already occurred.

2. `gsc_clicks` — February search clicks; knowable at the decision moment because the complete February feature window has already occurred.

3. `gsc_avg_position` — February average search position; knowable at the decision moment because it comes from the completed February feature window.

4. `word_count` — content length; knowable at the decision moment because it is a property of the content available before the decision cutoff.

5. `content_created_date` — content creation date; knowable at the decision moment because the creation date exists before the future March label window.

In [19]:
# Small feature frame for February 2026

# Final small five-feature frame for February 2026

features = con.sql(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position,
    d.word_count,
    d.content_created_date
FROM {FEB} AS f
LEFT JOIN {DIM} AS d
    ON f.client_hash_id = d.client_hash_id
    AND f.content_hash_id = d.content_hash_id
WHERE f.gsc_data_available IS TRUE
LIMIT 20
""").df()

display(features)

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_created_date
0,client_e547b89c05043229,content_7995404695ee1ffd,57,0,31.192982,2828,2025-07-17
1,client_e547b89c05043229,content_1eea820697c3b95a,13,0,6.538462,2613,2025-07-17
2,client_e547b89c05043229,content_ccbb253f142217c3,59,0,16.966102,2689,2025-07-17
3,client_e547b89c05043229,content_ae16a6b9cf64c80a,17,0,16.882353,2928,2025-07-17
4,client_e547b89c05043229,content_acf700633f016e5a,6,0,4.500000,3108,2025-07-17
5,client_e547b89c05043229,content_9abd8b303f805847,27,1,6.925926,2992,2025-07-17
6,client_e547b89c05043229,content_5f58c55cbfee172a,10,0,24.600000,2225,2025-07-17
7,client_e547b89c05043229,content_712e44562fed8ff2,5,0,18.200000,2288,2025-07-17
8,client_e547b89c05043229,content_6fe390ba3af1e456,81,0,31.740741,2797,2025-07-17
9,client_e547b89c05043229,content_3ad5d2160242b9ca,24,0,3.291667,2396,2025-07-17


In [20]:
# Leakage experiment: build the March label from the future label window

label_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS march_clicks,
    SUM(gsc_impressions) AS march_impressions
FROM {MAR}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 100
""").df()

display(label_df.head())
print("Label rows:", len(label_df))

,client_hash_id,content_hash_id,march_clicks,march_impressions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.0,6523.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,0.0,453.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,6.0,5630.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,13.0,4944.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,1.0,429.0


Label rows: 101441


In [21]:
# Deliberate leakage experiment
# March CTR is future label information and must NOT be used as a real feature.

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Create observed March CTR
label_df["march_ctr"] = (
    label_df["march_clicks"] / label_df["march_impressions"]
)

# Create a simple binary label:
# 1 = March CTR is above the median among eligible pages
ctr_cutoff = label_df["march_ctr"].median()
label_df["label"] = (label_df["march_ctr"] > ctr_cutoff).astype(int)

# Join February features with the March outcome
leak_df = features.merge(
    label_df[["client_hash_id", "content_hash_id", "march_ctr", "label"]],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Keep only rows with complete values
leak_df = leak_df.dropna(
    subset=["gsc_impressions", "gsc_clicks", "gsc_avg_position", "word_count", "label"]
)

# Deliberately leaked feature: march_ctr
X = leak_df[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "word_count",
    "march_ctr"
]]

y = leak_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Deliberate leakage experiment")
print("Test accuracy with future March CTR included:",
      round(accuracy_score(y_test, pred), 4))
print("Leaked feature: march_ctr")

Deliberate leakage experiment
Test accuracy with future March CTR included: 0.8333
Leaked feature: march_ctr


In [22]:
# Remove the deliberately leaked future feature
# and evaluate using February-only features.

honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "word_count"
]

X_honest = leak_df[honest_features]
y = leak_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_accuracy = accuracy_score(y_test, honest_pred)

print("Honest experiment")
print("Future March CTR removed: YES")
print("Features used:", honest_features)
print("Test accuracy without leakage:",
      round(honest_accuracy, 4))

Honest experiment
Future March CTR removed: YES
Features used: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'word_count']
Test accuracy without leakage: 0.3333


### Leakage lesson

I deliberately added `march_ctr` from the future March label window to the February feature frame. The test accuracy was 0.8333 with this leaked feature.

After removing `march_ctr`, the test accuracy was 0.3333 using only February-available features.

This shows that a future label-derived field can create a large artificial performance jump. `march_ctr` is therefore removed from the final feature set and is not used as a legitimate ranking feature.

The honest feature set uses information available before the March label window.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verification 1: grain
# Expected grain: one row per report_date + client_hash_id + content_hash_id

q1 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain_rows
FROM {MAR}
"""

display(con.sql(q1).df())

,total_rows,distinct_grain_rows
0,9841378,9841378


In [24]:
# Verification 2: March slice row count and date span

q2 = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {MAR}
"""

display(con.sql(q2).df())

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [25]:
# Verification 3: GSC availability
# Use IS TRUE explicitly as required by the assignment.

q3 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
FROM {MAR}
"""

display(con.sql(q3).df())

,total_rows,gsc_available_rows
0,9841378,3611061


## 4. Data limits

### 4. Data limits

- The warehouse does not provide a perfectly balanced history across all content records, so results may be affected by uneven data coverage.
- Some records have limited or unavailable GSC data, so search-performance analysis cannot represent every content record equally.
- The February feature window and March label window are separate but adjacent; the model cannot use information that becomes available only during the March outcome window.
- Historical optimization or update events may not always be safely available as-of the February decision cutoff, so they are excluded unless their timing can be verified.
- The analysis is based on observed warehouse signals and supports ranking/review prioritization; it does not establish that a feature causes a change in search performance.

### Named limitation

**GSC availability limitation:** Only records with available GSC data can contribute to the GSC-based signals used in this analysis. Records without GSC availability may therefore be underrepresented.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.